# ExpenseIQ - EDA & model training walk-through
Companion Jupyter notebook (the report lists Jupyter as a project tool).
Run from the repository root: `jupyter notebook notebooks/eda_and_training.ipynb`

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from backend.services.feature_engineering import (
    build_daily_series, generate_time_features, make_training_set, winsorize_anomalies)
from scripts.train_model import load_daily_series, build_models, evaluate

df = pd.read_csv('../data/transactions.csv', parse_dates=['date'])
df.head()

In [ ]:
monthly = df.assign(month=df['date'].dt.to_period('M')).groupby(['user_email', 'month'])['amount'].sum().unstack(0)
monthly.plot(figsize=(11, 4), title='Monthly spend per user (multi-year synthetic dataset)')
plt.ylabel('total spend'); plt.tight_layout(); plt.show()

In [ ]:
# Seasonality: average spend by calendar month (Diwali block in Oct/Nov is visible)
df.groupby(df['date'].dt.month)['amount'].sum().plot(kind='bar', figsize=(9, 3.5), color='#1a7f5a')
plt.title('Total spend by calendar month'); plt.tight_layout(); plt.show()

In [ ]:
# Feature engineering demo on one user's daily series
daily_all, daily_var = load_daily_series(Path('../data/transactions.csv'))
daily = winsorize_anomalies(daily_all[1], daily_var[1])
feats = generate_time_features(daily.tail(365))
feats[['lag_1', 'rolling_mean_7', 'rolling_mean_30', 'ema_7']].plot(figsize=(11, 4),
    title='Engineered features (last 365 days, demo user)')
plt.tight_layout(); plt.show()

In [ ]:
# Build the training set (stationary ratio target) and compare the three models
X, y, meta = make_training_set(daily_all, daily_var)
print(X.shape, 'samples; features:', list(X.columns))
X.head()

In [ ]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error

order = np.argsort(meta['as_of'].values)
X, y, meta = X.iloc[order], y.iloc[order], meta.iloc[order]
split = int(len(X) * 0.8)
X_tr, X_te = X.iloc[:split], X.iloc[split:]
y_tr, y_te = y.iloc[:split], y.iloc[split:]
y_te_total = meta['y_total'].iloc[split:].to_numpy()
base_te = (X_te['rolling_mean_30'].clip(lower=1.0) * X_te['days_in_target_month']).to_numpy()

rows = []
for name, model in build_models(quick=True).items():
    model.fit(X_tr, y_tr)
    pred = model.predict(X_te) * base_te
    rows.append({'model': name, **evaluate(y_te_total, pred)})
pd.DataFrame(rows).set_index('model')

The production artefact (`ml_models/xgboost_expense_forecaster_v2.pkl`) is created by
`scripts/train_model.py`, which uses the full (non-quick) hyper-parameters, time-series
cross-validation and writes `ml_models/model_metrics.json`.